# Reconstruct natural gaps
Use the complete input year to reconstruct March. Earlier observations establish station history. Later observations provide context for blocks near the end of the request. These natural gaps have no measured target, so this notebook examines coverage and method flags.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from giano.data import load_input

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sample = root / "examples" / "noaa-2024"
torch.set_num_threads(2)
ds = load_input(sample / "observations.csv", sample / "stations.csv")

In [ ]:
from giano.fill import fill

out = fill(ds, "2024-03-01", "2024-04-01", local_context=True, station_reference=True)
variables = ("temperature", "wind_speed", "wind_direction")
pd.DataFrame(
    {v: pd.Series(out[f"{v}_method"].values.ravel()).value_counts() for v in variables}
).fillna(0).astype(int)

Select the station with the most reconstructed temperatures in March. Plot the measured values as points and only the reconstructed hours as crosses. The output retains both series so that later analyses can filter estimates by method or reference.

In [ ]:
added = out.temperature.isnull() & out.temperature_filled.notnull()
station = added.sum("time").idxmax().item()
series = out.sel(station=station)
ax = series.temperature.to_pandas().plot(style=".", label="Observed", alpha=0.6)
series.temperature_filled.where(series.temperature.isnull()).to_pandas().plot(
    ax=ax,
    style="x",
    label="Reconstructed",
    ylabel="Temperature (°C)",
    title=str(station),
)
ax.legend()
plt.show()

## Require a contemporary physical reference
With only three stations in this network, an outage leaves at most two neighbours. Giano requires three eligible donors for its spatial reference. Compare the flags to see which estimates depended on local history.

In [ ]:
strict = fill(ds, "2024-03-01", "2024-04-01", ["temperature"], contemporary_only=True)
pd.crosstab(
    out.temperature_method.values.ravel(),
    strict.temperature_method.values.ravel(),
    rownames=["Default method"],
    colnames=["Contemporary-only method"],
)